In [4]:
"""
harmonizome_raw_to_csv.py  (v3 — CORRECT)
==========================================
The Harmonizome gene_attribute_edges.txt files have this structure:

  Columns  : source | source_desc | source_id | target | target_desc | target_id | weight
  Row 0    : GeneSym| <desc>      | GeneID    | <attr> | <attr_id>   | <id2>     | weight
  Row 1+   : actual data

So:
  - source     = gene symbol  (Head)
  - target     = attribute name  (Tail name)
  - target_desc= attribute ID or secondary name  (Tail ID, often)
  - target_id  = tertiary ID (sometimes useful)
  - weight     = edge weight  (filter == 1.0)

Row 0 is a metadata/label row — we skip it and use source/target directly.

Usage:
    python harmonizome_raw_to_csv.py
"""

import os
import pandas as pd
from pathlib import Path

# ──────────────────────────────────────────────────────────
HARMONIZOME_ROOT = "/storage/Arushi/090526_EvoAge/kg_formation/data_collection/harmonizome"
OUTPUT_DIR       = os.path.join(HARMONIZOME_ROOT, "refine_harmonizome", "new_refine_harmonizome")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ──────────────────────────────────────────────────────────
def read_edges(folder_name):
    """
    Read gene_attribute_edges.txt and return clean DataFrame with:
        GeneSym | GeneID | target | target_desc | target_id | weight
    Row 0 (metadata label row) is dropped.
    """
    path = os.path.join(HARMONIZOME_ROOT, folder_name, "gene_attribute_edges.txt")
    if not os.path.exists(path):
        print(f"  [SKIP] Not found: {folder_name}")
        return None

    for enc in ["utf-8", "latin-1", "cp1252"]:
        try:
            df = pd.read_csv(path, sep="\t", low_memory=False,
                             encoding=enc, dtype=str)
            break
        except UnicodeDecodeError:
            continue
    else:
        print(f"  [ERROR] Cannot decode: {folder_name}")
        return None

    # Drop row 0 — it's the metadata/label row (e.g. "GeneSym", "GeneID", ...)
    df = df.iloc[1:].reset_index(drop=True)

    # Rename the fixed columns to standard names
    df = df.rename(columns={
        "source"     : "GeneSym",
        "source_id"  : "GeneID",
        "target"     : "target",
        "target_desc": "target_desc",
        "target_id"  : "target_id",
        "weight"     : "weight",
    })

    # Convert weight to numeric
    df["weight"] = pd.to_numeric(df["weight"], errors="coerce")

    return df


def save(df, filename):
    out = os.path.join(OUTPUT_DIR, filename)
    df.to_csv(out, index=False)
    print(f"  [SAVED] {filename}  ({len(df):,} rows)")


def read_and_filter(folder_name, weight_filter=True):
    df = read_edges(folder_name)
    if df is None:
        return None
    if weight_filter:
        df = df[df["weight"] == 1.0]
    return df


# ══════════════════════════════════════════════════════════
print("\n" + "="*60)
print("Processing Harmonizome raw datasets → intermediate CSVs")
print("="*60)

# ──────────────────────────────────────────────────────────
# 1. Gene–Gene (PPI)
#    source=GeneSym, target=GeneSym (partner)
# ──────────────────────────────────────────────────────────
print("\n── Gene–Gene (PPI) ──")

ppi_datasets = [
    ("BioGRID Protein-Protein Interactions",         "protein-protein interaction_BioGRID.csv"),
    ("DIP Protein-Protein Interactions",             "protein -protein associations_DIP.csv"),
    ("BIND Biomolecular Interactions",               "gene-gene assosiation_BIND.csv"),
    ("HumanCyc Biomolecular Interactions",           "gene-gene assosiation_HumanCyc.csv"),
    ("PANTHER Biomolecular Interactions",            "gene-gene assosiation_PANTHER.csv"),
    ("PID Biomolecular Interactions",                "gene-gene assosiation_PID.csv"),
    ("Reactome Biomolecular Interactions",           "gene-gene assosiation_Reactome.csv"),
    ("HPRD Protein-Protein Interactions",            "gene-gene assosiation_HPRD.csv"),
    ("Pathway Commons Protein-Protein Interactions", "gene-gene assosiation_PathwayCommons.csv"),
    ("IntAct Biomolecular Interactions",             "gene-gene assosiation_IntAct.csv"),
    ("KEGG Biomolecular Interactions",               "gene-gene assosiation_KEGG.csv"),
    ("Hub Proteins Protein-Protein Interactions",    "gene-gene assosiation_HubProteins.csv"),
    ("NURSA Protein-Protein Interactions",           "gene-gene assosiation_NURSA.csv"),
    ("PhosphoSitePlus Substrates of Kinases",        "gene-gene assosiation_PhosphoSitePlus_Kinase.csv"),
    ("KEA Substrates of Kinases",                    "gene-gene assosiation_KEA.csv"),
]
for folder, outfile in ppi_datasets:
    df = read_and_filter(folder)
    if df is None: continue
    # target = partner GeneSym
    out = pd.DataFrame({
        "GeneSym"  : df["GeneSym"],
        "weight"   : df["weight"],
        "GeneSym_1": df["target"],   # partner gene symbol
        "relation" : "interacts_with",
        "Source"   : folder,
    })
    save(out, outfile)

# ──────────────────────────────────────────────────────────
# 2. Gene–Disease
#    source=GeneSym, target=Disease name, target_desc=DOID
# ──────────────────────────────────────────────────────────
print("\n── Gene–Disease ──")

disease_datasets = [
    ("DISEASES Curated Gene-Disease Assocation Evidence Scores",
     "gene-disease associations_DISEASES.csv"),
    ("DISEASES Experimental Gene-Disease Assocation Evidence Scores",
     "gene-disease associations_DISEASES.csv_1.csv"),
    ("DISEASES Text-mining Gene-Disease Assocation Evidence Scores",
     "gene-disease associations_DISEASES.csv_2.csv"),
    ("GAD Gene-Disease Associations",
     "gene-disease associations_GAD.csv"),
    ("GAD High Level Gene-Disease Associations",
     "gene-disease associations_GAD_high.csv"),
    ("GWASdb SNP-Disease Associations",
     "gene-disease associations_GWASdb.csv"),
    ("PhosphoSitePlus Phosphosite-Disease Associations",
     "gene-disease associations_PhosphoSitePlus.csv"),
    ("CTD Gene-Disease Associations",
     "gene-disease associations_CTD.csv"),
    ("OMIM Gene-Disease Associations",
     "gene-disease associations_OMIM.csv"),
    ("HPO Gene-Disease Associations",
     "gene-disease associations_HPO.csv"),
    ("dbGAP Gene-Trait Associations",
     "gene-disease associations_dbGAP.csv"),
]
for folder, outfile in disease_datasets:
    df = read_and_filter(folder)
    if df is None: continue
    out = pd.DataFrame({
        "GeneSym" : df["GeneSym"],
        "weight"  : df["weight"],
        "Disease" : df["target"],        # disease name
        "DOID"    : df["target_desc"],   # DOID or other ID
        "Source"  : folder,
    })
    save(out, outfile)

# GEO disease perturbations — target_desc has Disease name
df = read_and_filter("GEO Signatures of Differentially Expressed Genes for Diseases")
if df is not None:
    out = pd.DataFrame({
        "GeneSym": df["GeneSym"],
        "weight" : df["weight"],
        "Disease": df["target_desc"],   # target_desc = Disease (from row0 label)
        "Source" : "GEO",
    })
    save(out, "gene-disease perturbation associations_GEO .csv")

# HuGE Navigator — target=Disease, target_desc=UMLS CUI
df = read_and_filter("HuGE Navigator Gene-Phenotype Associations")
if df is not None:
    out = pd.DataFrame({
        "GeneSym" : df["GeneSym"],
        "weight"  : df["weight"],
        "Disease" : df["target"],
        "UMLS CUI": df["target_desc"],
        "Source"  : "HuGE Navigator",
    })
    save(out, "gene-phenotype associations_HuGE Navigator.csv")

# ──────────────────────────────────────────────────────────
# 3. Gene–Tissue
#    source=GeneSym, target=tissue name (or BTO ID)
# ──────────────────────────────────────────────────────────
print("\n── Gene–Tissue ──")

# GTEx Tissue Gene Expression — target=Tissue
df = read_and_filter("GTEx Tissue Gene Expression Profiles")
if df is not None:
    out = pd.DataFrame({"GeneSym": df["GeneSym"], "weight": df["weight"],
                        "Tissue": df["target"], "Source": "GTEx"})
    save(out, "gene_tissue_biogrkan.csv")

# Allen Brain Adult Human — target=Structure
df = read_and_filter("Allen Brain Atlas Adult Human Brain Tissue Gene Expression Profiles")
if df is not None:
    out = pd.DataFrame({"GeneSym": df["GeneSym"], "weight": df["weight"],
                        "Structure": df["target"], "Source": "Allen Brain Adult Human"})
    save(out, "gene-tisue_Allen Brain Atlas.csv")

# Allen Brain Adult Mouse — target=StructureName
df = read_and_filter("Allen Brain Atlas Adult Mouse Brain Tissue Gene Expression Profiles")
if df is not None:
    out = pd.DataFrame({"GeneSym": df["GeneSym"], "weight": df["weight"],
                        "StructureName": df["target"], "Source": "Allen Brain Adult Mouse"})
    save(out, "gene-tissue_Allen Brain Atlas.csv")

# GTEx Tissue Sample — target=Tissue Sample, target_desc=Tissue
df = read_and_filter("GTEx Tissue Sample Gene Expression Profiles")
if df is not None:
    out = pd.DataFrame({"GeneSym": df["GeneSym"], "weight": df["weight"],
                        "Tissue": df["target_desc"], "Source": "GTEx Sample"})
    save(out, "gene-tissue sample associations_GTEx.csv")

# Allen Brain Developing Microarray — target=Structure_Age_Gender_DonorID
df = read_and_filter("Allen Brain Atlas Developing Human Brain Tissue Gene Expression Profiles by Microarray")
if df is not None:
    out = pd.DataFrame({"GeneSym": df["GeneSym"], "weight": df["weight"],
                        "Structure_Age_Gender_DonorID": df["target"],
                        "Source": "Allen Brain Developing Microarray"})
    save(out, "gene-tissue sample associations_Allen Brain Atlas.csv_1.csv")

# TISSUES datasets — target=Tissue/Cell Type, target_desc=BTO
for suffix, outfile in [
    ("TISSUES Curated Tissue Protein Expression Evidence Scores",
     "gene-tissue associations_TISSUES .csv"),
    ("TISSUES Experimental Tissue Protein Expression Evidence Scores",
     "gene-tissue associations_TISSUES .csv_1.csv"),
    ("TISSUES Text-mining Tissue Protein Expression Evidence Scores",
     "gene-tissue associations_TISSUES .csv_2.csv"),
]:
    df = read_and_filter(suffix)
    if df is None: continue
    out = pd.DataFrame({"GeneSym": df["GeneSym"], "weight": df["weight"],
                        "BTO": df["target_desc"],          # BTO ID
                        "Tissue": df["target"],            # Tissue name
                        "Source": suffix})
    save(out, outfile)

# HPA Tissue Gene Expression — target=Tissue
df = read_and_filter("HPA Tissue Gene Expression Profiles")
if df is not None:
    out = pd.DataFrame({"GeneSym": df["GeneSym"], "weight": df["weight"],
                        "Tissue": df["target"], "Source": "HPA Tissue"})
    save(out, "gene-tissue associations_HPA.csv")

# HPA Tissue Sample — target=Tissue Sample, target_desc=Tissue
df = read_and_filter("HPA Tissue Sample Gene Expression Profiles")
if df is not None:
    out = pd.DataFrame({"GeneSym": df["GeneSym"], "weight": df["weight"],
                        "Tissue": df["target_desc"], "Source": "HPA Tissue Sample"})
    save(out, "gene-tissue associations_HPA.csv_1.csv")

# Allen Brain Prenatal — target=Structure
df = read_and_filter("Allen Brain Atlas Prenatal Human Brain Tissue Gene Expression Profiles")
if df is not None:
    out = pd.DataFrame({"GeneSym": df["GeneSym"], "weight": df["weight"],
                        "Structure": df["target"], "Source": "Allen Brain Prenatal"})
    save(out, "gene-tissue associations_Allen Brain Atlas.csv")

# TCGA — target=Cancer Name_Cancer Acronym_TCGA Barcode, target_desc=Cancer Name_Cancer Acronym
df = read_and_filter("Allen Brain Atlas Developing Human Brain Tissue Gene Expression Profiles by RNA-seq")
if df is not None:
    out = pd.DataFrame({"GeneSym": df["GeneSym"], "weight": df["weight"],
                        "Cancer Name_Cancer Acronym": df["target_desc"],
                        "Source": "Allen Brain RNA-seq"})
    save(out, "gene-tissue sample associations_TCGA .csv")

# BioGPS Human — target=Tissue
df = read_and_filter("BioGPS Human Cell Type and Tissue Gene Expression Profiles")
if df is not None:
    out = pd.DataFrame({"GeneSym": df["GeneSym"], "weight": df["weight"],
                        "Tissue": df["target"], "Source": "BioGPS Human"})
    save(out, "gene-tissue associations_BioGPS_Human.csv")

# BioGPS Mouse — target=Tissue
df = read_and_filter("BioGPS Mouse Cell Type and Tissue Gene Expression Profiles")
if df is not None:
    out = pd.DataFrame({"GeneSym": df["GeneSym"], "weight": df["weight"],
                        "Tissue": df["target"], "Source": "BioGPS Mouse"})
    save(out, "gene-tissue associations_BioGPS_Mouse.csv")

# Allen Brain adult — one more version
df = read_and_filter("Allen Brain Atlas Adult Human Brain Tissue Gene Expression Profiles")
if df is not None:
    out = pd.DataFrame({"GeneSym": df["GeneSym"], "weight": df["weight"],
                        "Structure": df["target"], "Source": "Allen Brain Adult Human"})
    save(out, "gene-tissue associations_Allen Brain Atlas.csv")

# ──────────────────────────────────────────────────────────
# 4. Gene–Cellular Component
#    target=GO Phrase (name), target_desc=GO ID
# ──────────────────────────────────────────────────────────
print("\n── Gene–Cellular Component ──")

cc_datasets = [
    ("COMPARTMENTS Curated Protein Localization Evidence Scores",
     "gene-cellular component associations_COMPARTMENTS.csv"),
    ("GO Cellular Component Annotations",
     "gene-cellular component associations_GO.csv"),
    ("LOCATE Curated Protein Localization Annotations",
     "gene-cellular component associations_LOCATE.csv"),
    ("LOCATE Predicted Protein Localization Annotations",
     "gene-cellular component associations_LOCATE.csv_1.csv"),
]
for folder, outfile in cc_datasets:
    df = read_and_filter(folder)
    if df is None: continue
    out = pd.DataFrame({
        "GeneSym"        : df["GeneSym"],
        "weight"         : df["weight"],
        "GO"             : df["target_desc"],   # GO ID  e.g. GO:0005634
        "GO_phrase"      : df["target"],        # GO name e.g. "nucleus"
        "relation_source": df["target_desc"],   # alias used by notebook
        "Source"         : folder,
    })
    save(out, outfile)

# ──────────────────────────────────────────────────────────
# 5. Gene–Phenotype
#    GWASdb: target=Phenotype, target_desc=HPOID
#    ClinVar/GWAS Catalog: target=Phenotype name
# ──────────────────────────────────────────────────────────
print("\n── Gene–Phenotype ──")

# GWASdb Phenotype — target_desc=HPOID
df = read_and_filter("GWASdb SNP-Phenotype Associations")
if df is not None:
    out = pd.DataFrame({"GeneSym": df["GeneSym"], "weight": df["weight"],
                        "HPOID": df["target_desc"], "Phenotype": df["target"],
                        "Source": "GWASdb"})
    save(out, "gene-phenotype associations_GWASdb.csv")

# ClinVar — target=Phenotype, target_desc=PhenotypeDatabase_PhenotypeID
df = read_and_filter("ClinVar SNP-Phenotype Associations")
if df is not None:
    out = pd.DataFrame({"GeneSym": df["GeneSym"], "weight": df["weight"],
                        "Phenotype": df["target"], "Source": "ClinVar"})
    save(out, "gene-phenotype associations_ClinVar.csv")

# GWAS Catalog — target=Phenotype
df = read_and_filter("GWAS Catalog SNP-Phenotype Associations")
if df is not None:
    out = pd.DataFrame({"GeneSym": df["GeneSym"], "weight": df["weight"],
                        "Phenotype": df["target"], "Source": "GWAS Catalog"})
    save(out, "gene-phenotype associations_GWAS Catalog.csv")

# HPO/MPO — target=Phenotype, target_desc=HPOID
df = read_and_filter("MPO Gene-Phenotype Associations")
if df is not None:
    out = pd.DataFrame({"GeneSym": df["GeneSym"], "weight": df["weight"],
                        "HPOID": df["target_desc"], "Phenotype": df["target"],
                        "Source": "MPO"})
    save(out, "gene-phenotype associations_HPO.csv")

# ──────────────────────────────────────────────────────────
# 6. Gene–Chemical / Drug
#    CTD: target=Chemical, target_desc=PubchemCID
#    DrugBank: target=Drug, target_desc=DrugBankID
# ──────────────────────────────────────────────────────────
print("\n── Gene–Chemical ──")

df = read_and_filter("CTD Gene-Chemical Interactions")
if df is not None:
    out = pd.DataFrame({"GeneSym": df["GeneSym"], "weight": df["weight"],
                        "Chemical": df["target"], "PubchemCID": df["target_desc"],
                        "Source": "CTD"})
    save(out, "gene-chemical associations_CTD.csv")

df = read_and_filter("DrugBank Drug Targets")
if df is not None:
    out = pd.DataFrame({"GeneSym": df["GeneSym"], "weight": df["weight"],
                        "DrugName": df["target"], "DrugBankID": df["target_desc"],
                        "Source": "DrugBank"})
    save(out, "gene-drug associations_DrugBank.csv")

df = read_and_filter("Guide to Pharmacology Chemical Ligands of Receptors")
if df is not None:
    out = pd.DataFrame({"GeneSym": df["GeneSym"], "weight": df["weight"],
                        "DrugName": df["target"], "DrugBankID": df["target_desc"],
                        "Source": "Guide to Pharmacology"})
    save(out, "gene-drug associations_GtP.csv")

# ──────────────────────────────────────────────────────────
# 7. Gene–Metabolite
#    HMDB: target=Metabolite name, target_desc=HMDB ACC (ID)
# ──────────────────────────────────────────────────────────
print("\n── Gene–Metabolite ──")

df = read_and_filter("HMDB Metabolites of Enzymes")
if df is not None:
    out = pd.DataFrame({"GeneSym": df["GeneSym"], "weight": df["weight"],
                        "Metabolite": df["target"],
                        "HMDB ACC.1": df["target_desc"],   # HMDB ACC ID
                        "Source": "HMDB"})
    save(out, "gene-metabolite associations_HMDB.csv")

# ──────────────────────────────────────────────────────────
# 8. Gene–Protein (interacting proteins)
#    target=partner GeneSym, need to map GeneSym → UniProt in notebook
# ──────────────────────────────────────────────────────────
print("\n── Gene–Protein ──")

protein_datasets = [
    ("NURSA Protein Complexes",
     "gene-interacting protein associations_NURSA.csv"),
    ("Pathway Commons Protein-Protein Interactions",
     "gene-interacting protein associations_Pathway Commons.csv"),
    ("Recon X Predicted Biomolecular Interactions",
     "gene-interacting protein associations_ReconX.csv"),
    ("Guide to Pharmacology Protein Ligands of Receptors",
     "gene-ligand (protein) associations_Guide to Pharmacology.csv"),
]
for folder, outfile in protein_datasets:
    df = read_and_filter(folder)
    if df is None: continue
    out = pd.DataFrame({
        "GeneSym"  : df["GeneSym"],
        "weight"   : df["weight"],
        "GeneSym_1": df["target"],   # partner gene symbol (notebook maps to UniProt)
        "Source"   : folder,
    })
    save(out, outfile)

# ──────────────────────────────────────────────────────────
# 9. Gene–Pathway
#    target=Pathway name, target_desc varies
# ──────────────────────────────────────────────────────────
print("\n── Gene–Pathway ──")

df = read_and_filter("Reactome Pathways")
if df is not None:
    out = pd.DataFrame({"GeneSym": df["GeneSym"], "weight": df["weight"],
                        "Pathway": df["target"], "PathwayID": df["target_id"],
                        "Source": "Reactome"})
    save(out, "gene-pathway associations_Reactome.csv")

# ──────────────────────────────────────────────────────────
# 10. Gene–Molecular Function / Biological Process
#     target=GO Phrase, target_desc=GO ID
# ──────────────────────────────────────────────────────────
print("\n── Gene–Molecular Function / Biological Process ──")

df = read_and_filter("GO Molecular Function Annotations")
if df is not None:
    out = pd.DataFrame({"GeneSym": df["GeneSym"], "weight": df["weight"],
                        "relation_source": df["target_desc"],  # GO ID
                        "Function": df["target"],              # GO phrase
                        "Source": "GO MF"})
    save(out, "gene-molecular function associations_GO.csv")

df = read_and_filter("GO Biological Process Annotations")
if df is not None:
    out = pd.DataFrame({"GeneSym": df["GeneSym"], "weight": df["weight"],
                        "relation_source": df["target_desc"],  # GO ID
                        "Function": df["target"],              # GO phrase
                        "Source": "GO BP"})
    save(out, "gene-biological process associations_GO.csv")

# ──────────────────────────────────────────────────────────
# 11. Gene–Gene (TF / co-expression)
#     target=partner GeneSym (TF or co-expressed gene)
# ──────────────────────────────────────────────────────────
print("\n── Gene–Gene (TF / co-expression) ──")

tf_datasets = [
    ("CHEA Transcription Factor Targets",
     "gene-gene associations_TRANSFAC .csv"),
    ("CHEA Transcription Factor Binding Site Profiles",
     "gene-gene associations_TRANSFAC .csv_1.csv"),
    ("MSigDB Cancer Gene Co-expression Modules",
     "gene-co-expressed gene associations_MSigDB.csv"),
    ("ENCODE Transcription Factor Targets",
     "gene-gene associations_ENCODE_TF.csv"),
    ("JASPAR Predicted Transcription Factor Targets",
     "gene-gene associations_JASPAR.csv"),
    ("TRANSFAC Curated Transcription Factor Targets",
     "gene-gene associations_TRANSFAC_curated.csv"),
    ("TRANSFAC Predicted Transcription Factor Targets",
     "gene-gene associations_TRANSFAC_predicted.csv"),
]
for folder, outfile in tf_datasets:
    df = read_and_filter(folder)
    if df is None: continue
    # For CHEA TF Binding: target=GeneSym-PMID-CellType-Organism, target_desc=GeneSym (TF)
    # For others: target=GeneSym directly
    gene_sym_col = (df["target_desc"]
                    if "CHEA Transcription Factor Binding" in folder
                    else df["target"])
    out = pd.DataFrame({
        "GeneSym"  : df["GeneSym"],
        "weight"   : df["weight"],
        "GeneSym_1": gene_sym_col,
        "relation" : "regulates",
        "Source"   : folder,
    })
    save(out, outfile)

print("\n" + "="*60)
print("DONE — all CSVs written to:")
print(f"  {OUTPUT_DIR}")
print("="*60)
print(f"\nTotal files written: check ls {OUTPUT_DIR}")



Processing Harmonizome raw datasets → intermediate CSVs

── Gene–Gene (PPI) ──
  [SAVED] protein-protein interaction_BioGRID.csv  (318,714 rows)
  [SAVED] protein -protein associations_DIP.csv  (15,720 rows)
  [SAVED] gene-gene assosiation_BIND.csv  (29,660 rows)
  [SAVED] gene-gene assosiation_HumanCyc.csv  (966 rows)
  [SAVED] gene-gene assosiation_PANTHER.csv  (42,484 rows)
  [SAVED] gene-gene assosiation_PID.csv  (26,178 rows)
  [SAVED] gene-gene assosiation_Reactome.csv  (229,924 rows)
  [SAVED] gene-gene assosiation_HPRD.csv  (120,644 rows)
  [SAVED] gene-gene assosiation_PathwayCommons.csv  (3,527,164 rows)
  [SAVED] gene-gene assosiation_IntAct.csv  (155,411 rows)
  [SAVED] gene-gene assosiation_KEGG.csv  (10 rows)
  [SAVED] gene-gene assosiation_HubProteins.csv  (58,327 rows)
  [SAVED] gene-gene assosiation_NURSA.csv  (4,258 rows)
  [SAVED] gene-gene assosiation_PhosphoSitePlus_Kinase.csv  (6,013 rows)
  [SAVED] gene-gene assosiation_KEA.csv  (12,161 rows)

── Gene–Disease ──

In [5]:
# Further processing is in 2_harmonizome.ipynb